In [1]:
import pytest
import ipytest
from spectral import *

In [2]:
ipytest.autoconfig()

In [5]:
%%ipytest

def test_gaussian_peak():
    # Gaussian should be 1.0 at exactly the peak
    wl = np.array([500e-9])
    res = SpectralPhysics._gaussian(wl, peak=500e-9, fwhm=20e-9)
    assert res[0] == pytest.approx(1.0)

def test_poly_response_masking():
    # Should be 0 outside the defined nm range
    wl = np.array([200e-9, 1300e-9]) # Out of bounds for 330-1090nm
    coeffs = np.ones(5)
    res = SpectralPhysics._poly_response(wl, coeffs, 330, 1090)
    assert np.all(res == 0)


def test_white_led_spectrum_shape():
    wl = np.linspace(300e-9, 1200e-9, 500)
    spec = SpectralPhysics.white_led_spectrum(wl)
    assert spec.shape == (500,)
    assert np.any(spec > 0)
    # Check that it has two distinct regions (blue and yellow)
    blue_region = spec[np.argmin(np.abs(wl - 470e-9))]
    gap_region = spec[np.argmin(np.abs(wl - 520e-9))]
    assert blue_region > gap_region

def test_sun_spectrum_normalized():
    wl = np.linspace(300e-9, 1200e-9, 1000)
    spec = SpectralPhysics.sun_spectrum(wl)
    # The sun peak (5800K) is around 500nm, which is in our range
    assert np.max(spec) == pytest.approx(1.0, abs = 1e-4)

# --- 3. Filter and Responsivity Logic ---

def test_get_filter_transmission():
    wl = np.array([500e-9, 900e-9])
    # VLC_PASS (320-720nm)
    vlc = SpectralPhysics.get_filter_transmission("VLC_PASS", wl)
    assert vlc[0] == 1.0
    assert vlc[1] == 0.0
    
    # ALL_PASS
    all_p = SpectralPhysics.get_filter_transmission("ALL_PASS", wl)
    assert np.all(all_p == 1.0)

def test_calculate_effective_responsivity_identity():
    # If source and detector are the same and filter is 1, responsivity logic
    # should essentially be checking that the ratio is ~1.0 (weighted by detector shape)
    def flat_source(wl): return np.ones_like(wl)
    def flat_detector(wl): return np.ones_like(wl)
    
    resp = SpectralPhysics.calculate_effective_responsivity(flat_source, flat_detector, "ALL_PASS")
    assert resp == pytest.approx(1.0)

def test_get_responsivity_by_name_lookup():
    # Test a common lookup configuration
    val = SpectralPhysics.get_responsivity_by_name("WLED2PD")
    assert isinstance(val, float)
    assert 0 < val < 1.0

def test_invalid_config_name():
    with pytest.raises(ValueError, match="Unknown configuration"):
        SpectralPhysics.get_responsivity_by_name("FAKE_SENSOR")

........                                                                                     [100%]
8 passed in 0.02s
